# Import modules


In [1]:
!python3 -m pip install pandas chromadb groq

  Using cached pandas-3.0.5-cp311-cp311-macosx_11_0_arm64.whl.metadata (79 kB)
  Using cached chromadb-1.5.9-cp39-abi3-macosx_11_0_arm64.whl.metadata (5.0 kB)
  Using cached numpy-2.4.6-cp311-cp311-macosx_14_0_arm64.whl.metadata (6.6 kB)
  Using cached build-1.5.0-py3-none-any.whl.metadata (5.7 kB)
  Using cached pydantic-2.13.4-py3-none-any.whl.metadata (109 kB)
  Using cached pydantic_settings-2.15.0-py3-none-any.whl.metadata (3.9 kB)
  Using cached pybase64-1.5.0-cp311-cp311-macosx_11_0_arm64.whl.metadata (11 kB)
  Using cached opentelemetry_api-1.44.0-py3-none-any.whl.metadata (1.4 kB)
  Using cached opentelemetry_exporter_otlp_proto_grpc-1.44.0-py3-none-any.whl.metadata (2.6 kB)
  Using cached opentelemetry_sdk-1.44.0-py3-none-any.whl.metadata (1.6 kB)
  Using cached tokenizers-0.23.1-cp310-abi3-macosx_11_0_arm64.whl.metadata (9.8 kB)
  Using cached pypika-0.51.1-py2.py3-none-any.whl.metadata (51 kB)
  Using cached tqdm-4.70.0-py3-none-any.whl.metadata (57 kB)
  Using cached impor

In [2]:
import os
import pandas as pd
import chromadb
from chromadb.utils import embedding_functions
from dotenv import load_dotenv
from groq import Groq

# Load environment variables

In [3]:
load_dotenv()

True

# Create Vector Database

In [4]:
client = chromadb.Client()

# Load content to Vector DB

In [6]:
!python3 -m pip install sentence_transformers

  Using cached sentence_transformers-5.7.0-py3-none-any.whl.metadata (18 kB)
  Using cached transformers-5.15.0-py3-none-any.whl.metadata (32 kB)
  Using cached torch-2.13.0-cp311-cp311-macosx_14_0_arm64.whl.metadata (39 kB)
  Using cached scikit_learn-1.9.0-cp311-cp311-macosx_12_0_arm64.whl.metadata (11 kB)
  Using cached scipy-1.17.1-cp311-cp311-macosx_14_0_arm64.whl.metadata (62 kB)
  Using cached regex-2026.7.19-cp311-cp311-macosx_11_0_arm64.whl.metadata (40 kB)
  Using cached tokenizers-0.22.2-cp39-abi3-macosx_11_0_arm64.whl.metadata (7.3 kB)
  Using cached safetensors-0.8.0-cp310-abi3-macosx_11_0_arm64.whl.metadata (4.2 kB)
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached narwhals-2.24.0-py3-none-any.whl.metadata (15 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached networkx-3.6.1-py3-none-any.whl.metadata (6.8 kB)
  Using cached mpmath-1.3.0-py3-none-a

In [8]:
embedder = embedding_functions.SentenceTransformerEmbeddingFunction(
            model_name="all-MiniLM-L6-v2"
        )

In [9]:
collection = client.get_or_create_collection(
            name="qa_chat",
            embedding_function=embedder,
            metadata={"hnsw:space": "cosine"},
        )

In [10]:
data = pd.read_csv("data.csv")
questions = data["question"].astype(str).tolist()
questions[:2]

['what Valaxy offering or who is valaxy?',
 'Who is the instructor giving training on AIML course?']

In [12]:
answers = [{"answer": a} for a in data["answer"].astype(str).tolist()]
answers[:3]

[{'answer': 'Valaxy provides instructor-led training programs that empower the next generation of AI engineers.'},
 {'answer': 'PR Reddy is an experienced instructor specializing in the AI/ML technology stack.'},
 {'answer': 'The course covers Python for Agentic AI and Statistical Machine Learning and Deep Learning and Neural Networks and Generative AI and Agentic AI Frameworks and AWS Bedrock and LLMs and and MCP Protocol and among others.'}]

In [13]:
ids = [f"id_{i}" for i in range(len(questions))]
ids[:3]

['id_0', 'id_1', 'id_2']

In [16]:
collection.upsert(documents=questions, metadatas=answers, ids=ids)

# Create Query

In [17]:
query = "how much is the AIML course fee?"

In [18]:
def _normalize_query(q: str) -> str:
        return q.strip().rstrip(" ?!.")
query = _normalize_query(query)

In [19]:
query

'how much is the AIML course fee'

In [21]:
q_emb = embedder([query])[0]

# Query the Vector Database

In [22]:
results = collection.query(query_embeddings=[q_emb], n_results=5)

In [23]:
results

{'ids': [['id_90', 'id_94', 'id_1', 'id_2', 'id_70']],
 'embeddings': None,
 'documents': [['What will be the AIML course fee?',
   'AIML course duration please?',
   'Who is the instructor giving training on AIML course?',
   'what are the top technologies covered in AIML course?',
   'How do I enroll in a course?']],
 'uris': None,
 'included': ['metadatas', 'documents', 'distances'],
 'data': None,
 'metadatas': [[{'answer': 'The regular fee is INR 30000. Contact the admin team for available discount coupons.'},
   {'answer': 'The total course duration is 90 days.'},
   {'answer': 'PR Reddy is an experienced instructor specializing in the AI/ML technology stack.'},
   {'answer': 'The course covers Python for Agentic AI and Statistical Machine Learning and Deep Learning and Neural Networks and Generative AI and Agentic AI Frameworks and AWS Bedrock and LLMs and and MCP Protocol and among others.'},
   {'answer': 'Create an account and browse available courses and and click ‘Enroll’ o

In [24]:
documents = results['documents'][0]

In [25]:
documents

['What will be the AIML course fee?',
 'AIML course duration please?',
 'Who is the instructor giving training on AIML course?',
 'what are the top technologies covered in AIML course?',
 'How do I enroll in a course?']

In [26]:
metadatas = results['metadatas'][0]

In [27]:
metadatas

[{'answer': 'The regular fee is INR 30000. Contact the admin team for available discount coupons.'},
 {'answer': 'The total course duration is 90 days.'},
 {'answer': 'PR Reddy is an experienced instructor specializing in the AI/ML technology stack.'},
 {'answer': 'The course covers Python for Agentic AI and Statistical Machine Learning and Deep Learning and Neural Networks and Generative AI and Agentic AI Frameworks and AWS Bedrock and LLMs and and MCP Protocol and among others.'},
 {'answer': 'Create an account and browse available courses and and click ‘Enroll’ or ‘Start Learning’.'}]

In [29]:
pairs = []
for d, m in zip(documents, metadatas):
    a = (m.get("answer", "") if isinstance(m, dict) else "")
    if d or a:
        pairs.append(f"Q: {d}\nA: {a}")

context = "\n\n".join(pairs).strip()
context

'Q: What will be the AIML course fee?\nA: The regular fee is INR 30000. Contact the admin team for available discount coupons.\n\nQ: AIML course duration please?\nA: The total course duration is 90 days.\n\nQ: Who is the instructor giving training on AIML course?\nA: PR Reddy is an experienced instructor specializing in the AI/ML technology stack.\n\nQ: what are the top technologies covered in AIML course?\nA: The course covers Python for Agentic AI and Statistical Machine Learning and Deep Learning and Neural Networks and Generative AI and Agentic AI Frameworks and AWS Bedrock and LLMs and and MCP Protocol and among others.\n\nQ: How do I enroll in a course?\nA: Create an account and browse available courses and and click ‘Enroll’ or ‘Start Learning’.'

# Create Prompt

In [30]:
prompt = f"""
    You are a helpful FAQ assistant. Use ONLY the context to answer.
    Rewrite the answer in clear, friendly language (not verbatim), and format it nicely.
    - If the question asks for "course content", "course index", or similar: present a clean, ordered outline.
    - If context includes lists, use bullets or numbered steps.
    - If dates, prices, or times appear, surface them clearly (you may bold them).
    - If multiple Q/A pairs are relevant, synthesize them into one concise answer.
    - At the end of the answer, mention the following line only if the user’s query is about pricing: Are you interested in joining the AIML course to become an AI Expert? I can check if there are any special discounts available for you!
    If the answer is not in the context, reply exactly: "I don't know".

    Question: {query}

    Context:
    {context}
    """

In [31]:
prompt

'\n    You are a helpful FAQ assistant. Use ONLY the context to answer.\n    Rewrite the answer in clear, friendly language (not verbatim), and format it nicely.\n    - If the question asks for "course content", "course index", or similar: present a clean, ordered outline.\n    - If context includes lists, use bullets or numbered steps.\n    - If dates, prices, or times appear, surface them clearly (you may bold them).\n    - If multiple Q/A pairs are relevant, synthesize them into one concise answer.\n    - At the end of the answer, mention the following line only if the user’s query is about pricing: Are you interested in joining the AIML course to become an AI Expert? I can check if there are any special discounts available for you!\n    If the answer is not in the context, reply exactly: "I don\'t know".\n\n    Question: how much is the AIML course fee\n\n    Context:\n    Q: What will be the AIML course fee?\nA: The regular fee is INR 30000. Contact the admin team for available di

# Query the LLM 

In [33]:
groq = Groq()
llm_answer = groq.chat.completions.create(
            model="openai/gpt-oss-120b",
            messages=[{"role": "user", "content": prompt}],
        )

In [80]:
llm_answer

ChatCompletion(id='chatcmpl-3828da10-755b-48e6-a9fb-1581fde14a53', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='**The regular fee for the AIML course is 30 INR**. However, you may be eligible for a personal discount. To find out, please **check with the Admin team for personal discount coupons**.\nAre you interested in joining the AIML course to become an AI Expert? I can check if there are any special discounts available for you!', role='assistant', executed_tools=None, function_call=None, reasoning=None, tool_calls=None))], created=1760386850, model='llama-3.3-70b-versatile', object='chat.completion', system_fingerprint='fp_155ab82e98', usage=CompletionUsage(completion_tokens=73, prompt_tokens=406, total_tokens=479, completion_time=0.179275916, prompt_time=0.035280737, queue_time=0.199867432, total_time=0.214556653), usage_breakdown=None, x_groq={'id': 'req_01k7fkq7bvfaqv7bcgnb1hjctp'}, service_tier='on_demand')

In [35]:
llm_answer.choices[0]

Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='The AIML course costs **₹30,000**. If you have any discount coupons, you can reach out to the admin team for details.\n\n**Are you interested in joining the AIML course to become an AI Expert? I can check if there are any special discounts available for you!**', role='assistant', annotations=None, executed_tools=None, function_call=None, reasoning='We need to answer question: "how much is the AIML course fee". Use context. Provide clear friendly language, bold dates/prices etc. At the end, because query is about pricing, need to add line: "Are you interested in joining the AIML course to become an AI Expert? I can check if there are any special discounts available for you!"\n\nWe must not include extra text. Provide answer.', tool_calls=None))

In [36]:
answer = llm_answer.choices[0].message.content

In [37]:
answer

'The AIML course costs **₹30,000**. If you have any discount coupons, you can reach out to the admin team for details.\n\n**Are you interested in joining the AIML course to become an AI Expert? I can check if there are any special discounts available for you!**'

# RAG Pipeline completed